### Model Training : 

In [2]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn for ML models and metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error

# TensorFlow/Keras for LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# For reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

Libraries imported successfully!
TensorFlow version: 2.21.0


In [5]:
flow_df = pd.read_csv("../data/processed/flow_150_data.csv")
flow_df.head()

,flow,avg_speed,density,Sunday,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,year
0,15,33.000000,0.454545,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2016
1,19,47.894737,0.396703,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2016
2,15,32.400000,0.462963,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2016
3,18,32.444444,0.554795,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2016
4,20,31.950000,0.625978,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2016


In [6]:
FLOW_INTERVAL = 150
SEEABLE_PAST = 240
PREDICT_IN_FUTURE = 15
N_SPLITS = 4

N_STEPS = SEEABLE_PAST * 60 // FLOW_INTERVAL
N_FUTURE = PREDICT_IN_FUTURE * 60 // FLOW_INTERVAL

print(f"Configuration:")
print(f"  - Using {N_STEPS} past intervals ({SEEABLE_PAST} minutes)")
print(f"  - Predicting {N_FUTURE} intervals ahead ({PREDICT_IN_FUTURE} minutes)")
print(f"  - Validation splits: {N_SPLITS}")

Configuration:
  - Using 96 past intervals (240 minutes)
  - Predicting 6 intervals ahead (15 minutes)
  - Validation splits: 4


In [7]:
def create_sequences(flow_df, n_steps, n_future):
    flow_values = np.array(flow_df['flow'])
    x, y = [], []
    for i in range(len(flow_values)):
        end_index = i + n_steps
        target_index = end_index + n_future
        if target_index >= len(flow_values):
            break
        x.append(flow_values[i:end_index])
        y.append(flow_values[target_index])

    x = np.array(x)
    y = np.array(y)

    x = x.reshape((x.shape[0], x.shape[1], 1))

    return x, y


x, y = create_sequences(flow_df, N_STEPS, N_FUTURE)
print(f"\nDataset created:")
print(f"  X shape: {x.shape} (samples, timesteps, features)")
print(f"  Y shape: {y.shape} (samples,)")


Dataset created:
  X shape: (52353, 96, 1) (samples, timesteps, features)
  Y shape: (52353,) (samples,)


In [8]:
def time_series_split(n_samples, n_splits):
    fold_size = n_samples // n_splits
    
    for i in range(n_splits):
        start = i * fold_size
        end = start + fold_size
        
        # 80% train, 20% test
        split_point = int(0.8 * (end - start)) + start
        
        train_idx = list(range(start, split_point))
        test_idx = list(range(split_point, end))
        
        yield train_idx, test_idx

# Test the split
print(f"Cross-validation with {N_SPLITS} splits:")
for i, (train_idx, test_idx) in enumerate(time_series_split(len(x), N_SPLITS)):
    print(f"  Fold {i+1}: Train {len(train_idx)} samples, Test {len(test_idx)} samples")

Cross-validation with 4 splits:
  Fold 1: Train 10470 samples, Test 2618 samples
  Fold 2: Train 10470 samples, Test 2618 samples
  Fold 3: Train 10470 samples, Test 2618 samples
  Fold 4: Train 10470 samples, Test 2618 samples


In [9]:
def evaluate_model(Y_true, Y_pred):
    # Ensure predictions are non-negative (flow can't be negative)
    Y_pred = np.maximum(Y_pred, 0)
    
    # RMSE
    rmse = np.sqrt(mean_squared_error(Y_true, Y_pred))
    
    # MAE
    mae = mean_absolute_error(Y_true, Y_pred)
    
    # Hit Ratio (trend prediction accuracy)
    # Did we correctly predict if flow goes up or down?
    correct = 0
    total = 0
    for i in range(N_FUTURE, len(Y_true)):
        actual_trend = Y_true[i] - Y_true[i - N_FUTURE]
        pred_trend = Y_pred[i] - Y_true[i - N_FUTURE]
        if actual_trend * pred_trend > 0:  # Same sign
            correct += 1
        total += 1
    
    hit_ratio = correct / total if total > 0 else 0
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        'Hit Ratio': hit_ratio
    }

print("Evaluation function ready!")

Evaluation function ready!


In [10]:
# Store results for all models
results = {}

def print_results(name, metrics, train_time):
    """Print model results in a nice format."""
    print(f"\n{'='*50}")
    print(f"{name} Results:")
    print(f"{'='*50}")
    print(f"  RMSE:      {metrics['RMSE']:.4f}")
    print(f"  MAE:       {metrics['MAE']:.4f}")
    print(f"  Hit Ratio: {metrics['Hit Ratio']*100:.2f}%")
    print(f"  Time:      {train_time:.2f}s")

#### Random Forest Model :

In [11]:
def random_forest_model(X, Y):
    """
    Random Forest regression model.
    """
    all_Y_true = []
    all_Y_pred = []
    total_time = 0
    
    # Flatten X for sklearn: (samples, features)
    X_flat = X.reshape(X.shape[0], X.shape[1])
    
    for fold, (train_idx, test_idx) in enumerate(time_series_split(len(X), N_SPLITS)):
        X_train, X_test = X_flat[train_idx], X_flat[test_idx]
        Y_train, Y_test = Y[train_idx], Y[test_idx]
        
        start_time = time.time()
        
        # Create and train model
        model = RandomForestRegressor(
            n_estimators=100,      # Number of trees
            max_depth=16,          # Maximum depth of trees
            random_state=42,
            n_jobs=-1              # Use all CPU cores
        )
        model.fit(X_train, Y_train)
        
        # Predict
        Y_pred = model.predict(X_test)
        
        total_time += time.time() - start_time
        
        all_Y_true.extend(Y_test)
        all_Y_pred.extend(Y_pred)
        
        print(f"  Fold {fold+1}/{N_SPLITS} complete")
    
    metrics = evaluate_model(np.array(all_Y_true), np.array(all_Y_pred))
    print_results("Random Forest", metrics, total_time)
    
    return metrics, total_time

print("Training Random Forest...")
results['Random Forest'], _ = random_forest_model(x, y)

Training Random Forest...
  Fold 1/4 complete
  Fold 2/4 complete
  Fold 3/4 complete
  Fold 4/4 complete

Random Forest Results:
  RMSE:      9.5063
  MAE:       6.8611
  Hit Ratio: 74.99%
  Time:      18.25s


#### XGBoost Model :

In [12]:
def xg_boost_model(X, Y):
    """
    XGBoost regression model.
    """
    import xgboost as xgb
    
    all_Y_true = []
    all_Y_pred = []
    total_time = 0
    
    # Flatten X for sklearn: (samples, features)
    X_flat = X.reshape(X.shape[0], X.shape[1])
    
    for fold, (train_idx, test_idx) in enumerate(time_series_split(len(X), N_SPLITS)):
        X_train, X_test = X_flat[train_idx], X_flat[test_idx]
        Y_train, Y_test = Y[train_idx], Y[test_idx]
        
        start_time = time.time()
        
        # Create and train model
        model = xgb.XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, Y_train)
        
        # Predict
        Y_pred = model.predict(X_test)
        
        total_time += time.time() - start_time
        
        all_Y_true.extend(Y_test)
        all_Y_pred.extend(Y_pred)
        
        print(f"  Fold {fold+1}/{N_SPLITS} complete")
    
    metrics = evaluate_model(np.array(all_Y_true), np.array(all_Y_pred))
    print_results("XGBoost", metrics, total_time)
    
    return metrics, total_time

results['XGBoost'], _ = xg_boost_model(x, y)

  Fold 1/4 complete
  Fold 2/4 complete
  Fold 3/4 complete
  Fold 4/4 complete

XGBoost Results:
  RMSE:      9.4384
  MAE:       6.8235
  Hit Ratio: 74.78%
  Time:      2.60s


#### LSTM Model :

In [13]:
def create_lstm_model(input_shape):
    """
    Create an LSTM model for time series prediction.
    
    Architecture:
    - LSTM layer with 50 units
    - Dense output layer (1 unit for regression)
    """
    model = Sequential([
        LSTM(50, activation='sigmoid', input_shape=input_shape),
        Dense(1)
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse'
    )
    
    return model

def lstm_model(X, Y):
    """
    LSTM regression model.
    """
    all_Y_true = []
    all_Y_pred = []
    total_time = 0
    
    for fold, (train_idx, test_idx) in enumerate(time_series_split(len(X), N_SPLITS)):
        X_train, X_test = X[train_idx], X[test_idx]
        Y_train, Y_test = Y[train_idx], Y[test_idx]
        
        start_time = time.time()
        
        # Create and train model
        model = create_lstm_model((X.shape[1], X.shape[2]))
        
        model.fit(
            X_train, Y_train,
            validation_split=0.2,  # Use 20% for validation
            epochs=15,             # Number of training epochs
            batch_size=64,         # Batch size
            verbose=0              # Suppress output
        )
        
        # Predict
        Y_pred = model.predict(X_test, verbose=0).flatten()
        
        total_time += time.time() - start_time
        
        all_Y_true.extend(Y_test)
        all_Y_pred.extend(Y_pred)
        
        print(f"  Fold {fold+1}/{N_SPLITS} complete")
    
    metrics = evaluate_model(np.array(all_Y_true), np.array(all_Y_pred))
    print_results("LSTM", metrics, total_time)
    
    return metrics, total_time

print("Training LSTM (this may take a few minutes)...")
results['LSTM'], _ = lstm_model(x, y)

Training LSTM (this may take a few minutes)...
  Fold 1/4 complete
  Fold 2/4 complete
  Fold 3/4 complete
  Fold 4/4 complete

LSTM Results:
  RMSE:      12.5181
  MAE:       8.9134
  Hit Ratio: 70.06%
  Time:      216.51s


#### Model Comparison :

In [14]:
# Create comparison table
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.round(4)
comparison_df['Hit Ratio'] = (comparison_df['Hit Ratio'] * 100).round(2).astype(str) + '%'

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string())
print("\nLower RMSE/MAE is better")
print("Higher Hit Ratio is better")


MODEL COMPARISON
                  RMSE     MAE Hit Ratio
Random Forest   9.5063  6.8611    74.99%
XGBoost         9.4384  6.8235    74.78%
LSTM           12.5181  8.9134    70.06%

Lower RMSE/MAE is better
Higher Hit Ratio is better


In [15]:
# Find the best model
best_model = min(results.items(), key=lambda x: x[1]['RMSE'])
print(f"\n Best Model (lowest RMSE): {best_model[0]}")
print(f"   RMSE: {best_model[1]['RMSE']:.4f}")
print(f"   MAE:  {best_model[1]['MAE']:.4f}")


 Best Model (lowest RMSE): XGBoost
   RMSE: 9.4384
   MAE:  6.8235
